# Privacy-Preserving Fraud Detection with FHE

## Motivation

Traditional fraud detection: bank receives customer data → runs model → predicts fraud.
**Problem:** bank sees all sensitive customer data (balance, transactions, etc.)

FHE approach: customer encrypts their data → bank runs model on encrypted data → 
returns encrypted result → only customer can decrypt.
**Result:** bank never sees real data, but still gets fraud prediction!

## What is Fully Homomorphic Encryption?

FHE allows computations directly on encrypted data without decrypting it.
The result, when decrypted, matches the result of the same computation on plaintext.

**This notebook demonstrates:**
1. Train logistic regression on fraud data (plaintext)
2. Encrypt customer features using TenSEAL (CKKS scheme)
3. Run inference on encrypted data
4. Compare results with plaintext model

## 1. Setup & Load Data

In [6]:
import tenseal as ts
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load fraud data
df = pd.read_csv('data/creditcard.csv')

print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['Class'].mean():.3%}")
print(f"\nFeatures: {df.columns.tolist()}")

Dataset shape: (284807, 31)
Fraud rate: 0.173%

Features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


## 2. Train Plaintext Logistic Regression

We use Logistic Regression because it's compatible with FHE —
it only requires addition and multiplication (linear operations).
Neural networks with ReLU activations are harder to implement in FHE.

In [9]:
# Prepare features
feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols].values
y = df['Class'].values

# Scale features — important for both LR and FHE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split — stratified to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, fraud: {y_train.sum()}")
print(f"Test:  {X_test.shape}, fraud: {y_test.sum()}")

# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)

# Evaluate
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

print(f"\n=== Plaintext Model Results ===")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Normal', 'Fraud'])}")

Train: (227845, 30), fraud: 394
Test:  (56962, 30), fraud: 98

=== Plaintext Model Results ===
ROC-AUC: 0.9721

              precision    recall  f1-score   support

      Normal       1.00      0.98      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



### Results

Plaintext Logistic Regression:
- **ROC-AUC: 0.9721** — excellent discrimination
- **Recall (Fraud): 0.92** — catches 92% of fraud cases
- Low precision (0.06) — expected with 0.17% fraud rate and balanced weights

This model will serve as our baseline.
Next: implement the same inference using FHE — encrypted inputs, same predictions.

## 3. FHE Setup — Encryption Context

We use the CKKS scheme — designed for approximate arithmetic on real numbers.
Perfect for ML inference where small approximation errors are acceptable.

**Key parameters:**
- `poly_modulus_degree` — security/performance tradeoff (8192 = 128-bit security)
- `coeff_mod_bit_sizes` — precision of computations
- `scale` — controls precision of encoded numbers

In [13]:
# Create TenSEAL CKKS context
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.generate_galois_keys()
context.global_scale = 2**40

print("FHE context created!")
print(f"Scheme: CKKS (approximate arithmetic)")
print(f"Security level: 128-bit")
print(f"Poly modulus degree: 8192")
print(f"Scale: 2^40")

# Take small sample for FHE demo (FHE is slow!)
n_samples = 10
X_fhe_sample = X_test[:n_samples]
y_fhe_sample = y_test[:n_samples]

print(f"\nSample for FHE inference: {n_samples} transactions")
print(f"Fraud in sample: {y_fhe_sample.sum()}")
print(f"\nOriginal (plaintext) features sample[0]:")
print(np.round(X_fhe_sample[0][:5], 4), "... (30 features total)")

# Encrypt the samples
print("\nEncrypting samples...")
enc_samples = [
    ts.ckks_vector(context, X_fhe_sample[i].tolist())
    for i in range(n_samples)
]
print(f"{n_samples} transactions encrypted!")
print(f"Encrypted vector type: {type(enc_samples[0])}")

FHE context created!
Scheme: CKKS (approximate arithmetic)
Security level: 128-bit
Poly modulus degree: 8192
Scale: 2^40

Sample for FHE inference: 10 transactions
Fraud in sample: 0

Original (plaintext) features sample[0]:
[ 1.3887 -0.3443  0.8527 -0.7325 -0.9382] ... (30 features total)

Encrypting samples...
10 transactions encrypted!
Encrypted vector type: <class 'tenseal.tensors.ckksvector.CKKSVector'>


## 4. FHE Inference — Prediction on Encrypted Data

The bank receives encrypted vectors and performs inference
without ever seeing the actual transaction data.

How logistic regression works with FHE:
1. Compute dot product: w · x_encrypted + b  (homomorphic operations)
2. Apply sigmoid approximation (polynomial)
3. Return encrypted result — only client can decrypt!

In [18]:
import time

# Extract model weights
weights = lr.coef_[0].tolist()
bias = lr.intercept_[0]

print(f"Model weights: {len(weights)} features")
print(f"Bias: {bias:.4f}")

# Sigmoid approximation for FHE
# Real sigmoid can't be computed directly on encrypted data
# We use polynomial approximation: 0.5 + 0.197x - 0.004x^3
def sigmoid_approx(x):
    return 0.5 + 0.197 * x - 0.004 * (x * x * x)

# FHE Inference
print("\nRunning FHE inference on encrypted data...")
print("(Bank never sees plaintext features!)\n")

fhe_predictions = []
fhe_probs = []
start_time = time.time()

for i, enc_x in enumerate(enc_samples):
    # Homomorphic dot product: w · x_encrypted
    enc_result = enc_x.dot(weights)
    
    # Add bias (plaintext + ciphertext operation)
    enc_result += bias
    
    # Decrypt result (only client can do this!)
    logit = enc_result.decrypt()[0]
    
    # Apply sigmoid approximation
    prob = sigmoid_approx(logit)
    prob = max(0, min(1, prob))  # clip to [0,1]
    
    fhe_predictions.append(1 if prob > 0.5 else 0)
    fhe_probs.append(prob)
    print(f"Transaction {i+1}: prob={prob:.4f} → {'FRAUD' if prob>0.5 else 'Normal'}")

elapsed = time.time() - start_time
print(f"\nFHE inference time: {elapsed:.2f}s for {n_samples} samples")
print(f"Average per transaction: {elapsed/n_samples:.3f}s")

Model weights: 30 features
Bias: -3.8908

Running FHE inference on encrypted data...
(Bank never sees plaintext features!)

Transaction 1: prob=0.0313 → Normal
Transaction 2: prob=0.0566 → Normal
Transaction 3: prob=1.0000 → FRAUD
Transaction 4: prob=0.0000 → Normal
Transaction 5: prob=0.9692 → FRAUD
Transaction 6: prob=0.0000 → Normal
Transaction 7: prob=0.5466 → FRAUD
Transaction 8: prob=0.0000 → Normal
Transaction 9: prob=0.0496 → Normal
Transaction 10: prob=0.0692 → Normal

FHE inference time: 0.14s for 10 samples
Average per transaction: 0.014s


## 5. Comparison: Plaintext vs FHE Inference

In [21]:
# Plaintext predictions on same sample
plaintext_probs = lr.predict_proba(X_fhe_sample)[:, 1]
plaintext_preds = lr.predict(X_fhe_sample)

print("=== Plaintext vs FHE Comparison ===\n")
print(f"{'#':<4} {'Plaintext Prob':>15} {'FHE Prob':>10} {'Match':>8} {'True Label':>12}")
print("-" * 55)

matches = 0
for i in range(n_samples):
    match = plaintext_preds[i] == fhe_predictions[i]
    if match:
        matches += 1
    print(f"{i+1:<4} {plaintext_probs[i]:>15.4f} {fhe_probs[i]:>10.4f} "
          f"{'✅' if match else '❌':>8} "
          f"{'FRAUD' if y_fhe_sample[i]==1 else 'Normal':>12}")

print(f"\nPrediction match rate: {matches}/{n_samples} ({matches/n_samples:.0%})")
print(f"\nMax probability difference: "
      f"{max(abs(plaintext_probs[i]-fhe_probs[i]) for i in range(n_samples)):.4f}")
print(f"Mean probability difference: "
      f"{np.mean([abs(plaintext_probs[i]-fhe_probs[i]) for i in range(n_samples)]):.4f}")

=== Plaintext vs FHE Comparison ===

#     Plaintext Prob   FHE Prob    Match   True Label
-------------------------------------------------------
1             0.0058     0.0313        ✅       Normal
2             0.0683     0.0566        ✅       Normal
3             0.0001     1.0000        ❌       Normal
4             0.0153     0.0000        ✅       Normal
5             0.9455     0.9692        ✅       Normal
6             0.0135     0.0000        ✅       Normal
7             0.0008     0.5466        ❌       Normal
8             0.0272     0.0000        ✅       Normal
9             0.0645     0.0496        ✅       Normal
10            0.0044     0.0692        ✅       Normal

Prediction match rate: 8/10 (80%)

Max probability difference: 0.9999
Mean probability difference: 0.1742


## 6. Improving FHE Accuracy — Better Sigmoid Approximation

The issue: our sigmoid approximation `0.5 + 0.197x - 0.004x^3` 
diverges for large |x| values (transaction 3 had large logit).

Solution: use a more accurate polynomial approximation
valid for a wider range of x values.

In [24]:
# Better sigmoid approximation — valid for wider range
# Based on minimax polynomial approximation
def sigmoid_approx_v2(x):
    # Clip extreme values first
    x = max(-8, min(8, x))
    # More accurate 5th degree polynomial
    return 0.5 + 0.21687 * x - 0.008191 * x**3 + 0.000196 * x**5

print("Running improved FHE inference...")
fhe_predictions_v2 = []
fhe_probs_v2 = []

for i, enc_x in enumerate(enc_samples):
    enc_result = enc_x.dot(weights)
    enc_result += bias
    logit = enc_result.decrypt()[0]
    
    prob = sigmoid_approx_v2(logit)
    prob = max(0, min(1, prob))
    
    fhe_predictions_v2.append(1 if prob > 0.5 else 0)
    fhe_probs_v2.append(prob)

# Compare
print(f"\n{'#':<4} {'Plaintext':>10} {'FHE v1':>10} {'FHE v2':>10} {'Match v2':>10}")
print("-" * 50)

matches_v2 = 0
for i in range(n_samples):
    match = plaintext_preds[i] == fhe_predictions_v2[i]
    if match:
        matches_v2 += 1
    print(f"{i+1:<4} {plaintext_probs[i]:>10.4f} {fhe_probs[i]:>10.4f} "
          f"{fhe_probs_v2[i]:>10.4f} {'✅' if match else '❌':>10}")

print(f"\nMatch rate v1: {8}/10 (80%)")
print(f"Match rate v2: {matches_v2}/10 ({matches_v2/n_samples:.0%})")
print(f"Mean prob diff v2: "
      f"{np.mean([abs(plaintext_probs[i]-fhe_probs_v2[i]) for i in range(n_samples)]):.4f}")

Running improved FHE inference...

#     Plaintext     FHE v1     FHE v2   Match v2
--------------------------------------------------
1        0.0058     0.0313     0.0000          ✅
2        0.0683     0.0566     0.0556          ✅
3        0.0001     1.0000     0.0000          ✅
4        0.0153     0.0000     0.0000          ✅
5        0.9455     0.9692     0.9656          ✅
6        0.0135     0.0000     0.0000          ✅
7        0.0008     0.5466     0.0000          ✅
8        0.0272     0.0000     0.0000          ✅
9        0.0645     0.0496     0.0498          ✅
10       0.0044     0.0692     0.0000          ✅

Match rate v1: 8/10 (80%)
Match rate v2: 10/10 (100%)
Mean prob diff v2: 0.0115


### Results

| Metric | FHE v1 | FHE v2 |
|--------|--------|--------|
| Match rate | 80% | **100%** |
| Mean prob diff | 0.1742 | **0.0115** |

**Key improvement:** 5th degree polynomial approximation handles
extreme logit values better than 3rd degree.
The approximation error of 0.0115 is negligible for fraud detection.

## 7. Summary & Privacy Guarantees

### Traditional vs FHE Approach

**Traditional approach:**
Customer → raw data → Bank server → prediction

❌ Bank sees everything: account balance, transaction history, spending patterns

**FHE approach:**
Customer → encrypted data → Bank server → encrypted result → Customer decrypts

✅ Bank sees nothing — only ciphertexts. Same prediction quality!

---

### Results

| Metric | Value |
|--------|-------|
| Plaintext ROC-AUC | 0.9721 |
| FHE match rate | 100% (10/10) |
| FHE probability difference | 0.0115 (negligible) |
| FHE inference speed | 0.014s per transaction |

---

### Technical Details

- **Scheme:** CKKS (approximate arithmetic on real numbers)
- **Security:** 128-bit
- **Library:** TenSEAL
- **Model:** Logistic Regression (naturally FHE-compatible — only linear operations)

---

### Limitations

- Only linear models work easily (Logistic Regression, linear SVM)
- Neural networks require polynomial approximation of activation functions
- Demonstrated on 10 samples — production requires hardware optimization

---

### Real-World Applications in Fintech

- **Cross-bank fraud detection** — multiple banks collaborate without sharing customer data
- **GDPR-compliant cloud inference** — send encrypted data to cloud ML services
- **Privacy-preserving credit scoring** — lender scores you without seeing raw financials
- **Secure multi-party analytics** — industry benchmarking without data exposure